In [1]:
import pandas as pd
df = pd.read_csv("saber_pro.csv")
print(df.shape)

(4645, 57)


In [2]:
import time
import pandas as pd

def consultar_acreditacion(snies, plantilla_url, selector, texto_buscado, sesion):
    """Devuelve 1 si la página del programa muestra 'texto_buscado', 0 si no, y None si no se pudo leer."""
    from bs4 import BeautifulSoup                                  # lector de HTML (ya viene en Colab)
    r = sesion.get(plantilla_url.format(snies=snies), timeout=30, headers={"User-Agent": "proyecto-academico/1.0"})
    if r.status_code != 200:
        return None                                                # la página no respondió
    nodos = BeautifulSoup(r.text, "html.parser").select(selector)
    if not nodos:
        return None                                                # no se encontró el selector: NO se adivina
    return int(any(texto_buscado.lower() in n.get_text(" ", strip=True).lower() for n in nodos))

def scraping_acreditacion(codigos, plantilla_url, selector, texto_buscado, pausa=1.0):
    """Recorre los códigos SNIES, espera entre consultas (buena educación con el servidor) y arma una tabla."""
    import requests
    sesion, filas = requests.Session(), []
    for c in codigos:
        filas.append({"ESTU_SNIES_PRGMACADEMICO": c,
                      "programa_acreditado": consultar_acreditacion(c, plantilla_url, selector, texto_buscado, sesion)})
        time.sleep(pausa)
    return pd.DataFrame(filas)

# --- EJEMPLO DE USO (déjalo comentado hasta validar la URL y el selector) ---
# codigos = sorted(df.ESTU_SNIES_PRGMACADEMICO.unique())[:5]          # prueba con 5 programas
# acred = scraping_acreditacion(codigos, "URL_CON_{snies}", "SELECTOR_CSS", "alta calidad")
# df = df.merge(acred, on="ESTU_SNIES_PRGMACADEMICO", how="left")
print("Funciones de scraping definidas (no se ejecutó ninguna consulta).")
print("Programas distintos (códigos SNIES) a consultar:", df.ESTU_SNIES_PRGMACADEMICO.nunique())

Funciones de scraping definidas (no se ejecutó ninguna consulta).
Programas distintos (códigos SNIES) a consultar: 469


In [5]:
prog = pd.read_excel("programas_snies.xlsx", sheet_name="Programas")
prog = prog[pd.to_numeric(prog["CÓDIGO_SNIES_DEL_PROGRAMA"], errors="coerce").notna()]
print(prog.shape)
print(prog["RECONOCIMIENTO_DEL_MINISTERIO"].value_counts())

(1, 39)
RECONOCIMIENTO_DEL_MINISTERIO
Registro calificado    1
Name: count, dtype: int64


/Users/mario/Documents/saber_pro.csv/.venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [6]:
prog = pd.read_excel("programas_snies.xlsx", sheet_name="Programas")
prog = prog[pd.to_numeric(prog["CÓDIGO_SNIES_DEL_PROGRAMA"], errors="coerce").notna()].copy()
prog["ESTU_SNIES_PRGMACADEMICO"] = pd.to_numeric(prog["CÓDIGO_SNIES_DEL_PROGRAMA"]).astype(int)

def marcar(x):
    if pd.isna(x):
        return None          # sin dato: no se adivina
    return int("alta calidad" in x.lower())

prog["programa_acreditado"] = prog["RECONOCIMIENTO_DEL_MINISTERIO"].apply(marcar)
acred = prog[["ESTU_SNIES_PRGMACADEMICO", "programa_acreditado"]]

df = df.merge(acred, on="ESTU_SNIES_PRGMACADEMICO", how="left")
print(df.shape)
print(df["programa_acreditado"].value_counts(dropna=False))
df.to_csv("saber_pro_con_acreditacion.csv", index=False)

/Users/mario/Documents/saber_pro.csv/.venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


(4645, 58)
programa_acreditado
1.0    2558
0.0    1706
NaN     381
Name: count, dtype: int64


In [7]:
mods = ["MOD_RAZONA_CUANTITAT_PUNT", "MOD_LECTURA_CRITICA_PUNT",
        "MOD_COMUNI_ESCRITA_PUNT", "MOD_INGLES_PUNT", "MOD_COMPETEN_CIUDADA_PUNT"]

d = df.drop_duplicates().drop_duplicates(subset="ESTU_CONSECUTIVO", keep="first")
d = d[d["ESTU_ESTADOINVESTIGACION"] == "PUBLICAR"]
d = d[~(d[mods].isna().any(axis=1) | (d[mods] == 0).any(axis=1))].copy()

d["promedio"] = d[mods].mean(axis=1)
d["riesgo"] = (d["promedio"] <= d["promedio"].quantile(0.25)).astype(int)
print(len(d))   # debe dar 4379

d["acreditacion"] = d["programa_acreditado"].map({1.0: "Acreditado", 0.0: "No acreditado"}).fillna("Sin dato")

print(d.groupby("acreditacion")["riesgo"].agg(["size", "mean"]).round(3))
print(d.groupby(["ESTU_METODO_PRGM", "acreditacion"])["riesgo"].agg(["size", "mean"]).round(3))

4379
               size   mean
acreditacion              
Acreditado     2412  0.180
No acreditado  1606  0.306
Sin dato        361  0.474
                                size   mean
ESTU_METODO_PRGM acreditacion              
DISTANCIA        Acreditado      160  0.431
                 No acreditado   476  0.504
                 Sin dato        227  0.573
DISTANCIA VITUAL Acreditado       52  0.596
                 No acreditado   239  0.464
                 Sin dato         39  0.462
PRESENCIAL       Acreditado     2200  0.151
                 No acreditado   891  0.157
                 Sin dato         95  0.242


In [2]:
%pip install pandas requests beautifulsoup4

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 24.7 MB/s  0:00:00eta 0:00:01
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 47.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [beautifulsoup4]m [pandas]
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel to use updated packages.
